In [1]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import joblib
from pathlib import Path

from surprise import (
    Dataset, Reader, SVD, SVDpp, NMF,
    accuracy, dump
)
from surprise.model_selection import (
    cross_validate, GridSearchCV,
    train_test_split as surprise_split
)

import mlflow
import mlflow.sklearn

from src.utils.config import settings

sns.set_theme(style="whitegrid")
Path('../../models/checkpoints').mkdir(
    parents=True, exist_ok=True)

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
# Setup MLflow 

mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)

print(f"✅ MLflow tracking URI : "
      f"{settings.MLFLOW_TRACKING_URI}")
print(f"   Experiment         : "
      f"{settings.MLFLOW_EXPERIMENT_NAME}")
print(f"   UI                 : "
      f"http://localhost:5000")

2026/06/02 17:57:32 INFO mlflow.tracking.fluent: Experiment with name 'recsys_experiments' does not exist. Creating a new experiment.


✅ MLflow tracking URI : http://localhost:5000
   Experiment         : recsys_experiments
   UI                 : http://localhost:5000


In [4]:
# Load Data + Prepare For Surprise
ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')
movies  = pd.read_csv(PROC + 'movies_master.csv',
                      low_memory=False)
movies  = movies[['movieId', 'title']].dropna(
    subset=['movieId'])
movies['movieId'] = movies['movieId'].astype(int)

# Surprise needs: userId, movieId, rating
# Rating scale 0.5 to 5.0
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

# Train/test split
trainset, testset = surprise_split(
    data, test_size=0.2, random_state=42)

print(f"Total ratings  : {len(ratings):,}")
print(f"Train ratings  : {trainset.n_ratings:,}")
print(f"Test ratings   : {len(testset):,}")
print(f"Unique users   : {trainset.n_users:,}")
print(f"Unique movies  : {trainset.n_items:,}")

Total ratings  : 100,004
Train ratings  : 80,003
Test ratings   : 20,001
Unique users   : 671
Unique movies  : 8,380


In [5]:
# Train SVD

print("Training SVD model...")
print("─" * 50)

with mlflow.start_run(run_name="SVD_baseline"):

    # Hyperparameters
    params = {
        "n_factors":   100,    # latent dimensions
        "n_epochs":    20,     # training iterations
        "lr_all":      0.005,  # learning rate
        "reg_all":     0.02,   # regularisation
        "random_state": 42,
    }

    mlflow.log_params(params)

    # Train
    start   = time.time()
    svd     = SVD(**params)
    svd.fit(trainset)
    elapsed = time.time() - start

    # Evaluate on test set
    predictions = svd.test(testset)
    rmse_score  = accuracy.rmse(predictions,
                                verbose=False)
    mae_score   = accuracy.mae(predictions,
                               verbose=False)

    mlflow.log_metric("rmse",     rmse_score)
    mlflow.log_metric("mae",      mae_score)
    mlflow.log_metric("train_time", elapsed)

    print(f"✅ SVD trained in {elapsed:.1f}s")
    print(f"   n_factors : {params['n_factors']}")
    print(f"   n_epochs  : {params['n_epochs']}")
    print(f"   RMSE      : {rmse_score:.4f}")
    print(f"   MAE       : {mae_score:.4f}")
    print(f"\n   Logged to MLflow ✅")

Training SVD model...
──────────────────────────────────────────────────
✅ SVD trained in 0.6s
   n_factors : 100
   n_epochs  : 20
   RMSE      : 0.9024
   MAE       : 0.6954

   Logged to MLflow ✅


In [7]:
# SVD Hyperparameter Tuning
print("SVD Hyperparameter Tuning (cross-validation)...")
print("This takes 3-5 minutes...\n")

param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs':  [20, 30],
    'lr_all':    [0.005, 0.010],
    'reg_all':   [0.02, 0.10],
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1,
    joblib_verbose=0,
)

start = time.time()
gs.fit(data)
elapsed = time.time() - start

print(f"✅ Grid search complete in {elapsed:.1f}s")
print(f"\nBest RMSE     : {gs.best_score['rmse']:.4f}")
print(f"Best params   : {gs.best_params['rmse']}")

# Train best SVD model
best_params = gs.best_params['rmse']
best_params['random_state'] = 42

with mlflow.start_run(run_name="SVD_tuned"):
    mlflow.log_params(best_params)

    svd_best = SVD(**best_params)
    svd_best.fit(trainset)

    predictions  = svd_best.test(testset)
    rmse_tuned   = accuracy.rmse(
        predictions, verbose=False)
    mae_tuned    = accuracy.mae(
        predictions, verbose=False)

    mlflow.log_metric("rmse", rmse_tuned)
    mlflow.log_metric("mae",  mae_tuned)

    print(f"\nTuned SVD RMSE : {rmse_tuned:.4f}")
    print(f"Tuned SVD MAE  : {mae_tuned:.4f}")
    print(f"Logged to MLflow ✅")


SVD Hyperparameter Tuning (cross-validation)...
This takes 3-5 minutes...

✅ Grid search complete in 6.7s

Best RMSE     : 0.8855
Best params   : {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

Tuned SVD RMSE : 0.8808
Tuned SVD MAE  : 0.6795
Logged to MLflow ✅


In [8]:
# SVD++ (Enhanced SVD)

print("Training SVD++ (enhanced SVD)...")
print("SVD++ also models implicit feedback")
print("─" * 50)

with mlflow.start_run(run_name="SVDpp"):

    params_pp = {
        "n_factors":    20,     # fewer factors — slower
        "n_epochs":     20,
        "lr_all":       0.007,
        "reg_all":      0.02,
        "random_state": 42,
    }

    mlflow.log_params(params_pp)

    start  = time.time()
    svdpp  = SVDpp(**params_pp)
    svdpp.fit(trainset)
    elapsed = time.time() - start

    predictions = svdpp.test(testset)
    rmse_pp     = accuracy.rmse(
        predictions, verbose=False)
    mae_pp      = accuracy.mae(
        predictions, verbose=False)

    mlflow.log_metric("rmse", rmse_pp)
    mlflow.log_metric("mae",  mae_pp)

    print(f"✅ SVD++ trained in {elapsed:.1f}s")
    print(f"   RMSE : {rmse_pp:.4f}")
    print(f"   MAE  : {mae_pp:.4f}")

Training SVD++ (enhanced SVD)...
SVD++ also models implicit feedback
──────────────────────────────────────────────────
✅ SVD++ trained in 17.7s
   RMSE : 0.8950
   MAE  : 0.6871


In [16]:
# ALS With Spark MLlib
import os
os.environ.setdefault(
    'JAVA_HOME',
    '/opt/homebrew/opt/openjdk@11'
)

from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import (
    RegressionEvaluator)
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("ALS-Recommender")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark session ready")

# Load ratings as Spark DataFrame
ratings_spark = spark.createDataFrame(ratings)

# Train/test split
train_spark, test_spark = ratings_spark.randomSplit(
    [0.8, 0.2], seed=42)

print(f"Train : {train_spark.count():,}")
print(f"Test  : {test_spark.count():,}")

✅ Spark session ready


Train : 80,164
Test  : 19,840


In [17]:
# Train ALS
print("Training ALS model...")
print("─" * 50)

with mlflow.start_run(run_name="ALS_Spark"):

    als_params = {
        "maxIter":        10,
        "rank":           50,    # latent factors
        "regParam":       0.01,  # regularisation
        "userCol":        "userId",
        "itemCol":        "movieId",
        "ratingCol":      "rating",
        "coldStartStrategy": "drop",
        "nonnegative":    False,
        "seed":           42,
    }

    mlflow.log_params({
        k: v for k, v in als_params.items()
        if k not in ['userCol','itemCol',
                     'ratingCol',
                     'coldStartStrategy']
    })

    start = time.time()
    als   = ALS(**als_params)
    als_model = als.fit(train_spark)
    elapsed   = time.time() - start

    # Evaluate
    predictions = als_model.transform(test_spark)
    predictions = predictions.dropna(
        subset=['prediction'])

    evaluator_rmse = RegressionEvaluator(
        metricName   = "rmse",
        labelCol     = "rating",
        predictionCol= "prediction"
    )
    evaluator_mae = RegressionEvaluator(
        metricName   = "mae",
        labelCol     = "rating",
        predictionCol= "prediction"
    )

    als_rmse = evaluator_rmse.evaluate(predictions)
    als_mae  = evaluator_mae.evaluate(predictions)

    mlflow.log_metric("rmse",       als_rmse)
    mlflow.log_metric("mae",        als_mae)
    mlflow.log_metric("train_time", elapsed)

    print(f"✅ ALS trained in {elapsed:.1f}s")
    print(f"   Rank     : {als_params['rank']}")
    print(f"   MaxIter  : {als_params['maxIter']}")
    print(f"   RMSE     : {als_rmse:.4f}")
    print(f"   MAE      : {als_mae:.4f}")
    print(f"   Logged to MLflow ✅")

Training ALS model...
──────────────────────────────────────────────────
✅ ALS trained in 2.0s
   Rank     : 50
   MaxIter  : 10
   RMSE     : 1.3763
   MAE      : 1.0698
   Logged to MLflow ✅


In [18]:
# ALS Recommendations

print("Generating ALS recommendations...")

# Top 10 recommendations for all users
user_recs = als_model.recommendForAllUsers(10)

# Pick sample user
sample_user = int(ratings['userId'].value_counts().index[0])

sample_recs = user_recs.filter(
    F.col("userId") == sample_user
).collect()

if sample_recs:
    rec_list = sample_recs[0]['recommendations']
    rec_df   = pd.DataFrame(
        rec_list, columns=['movieId', 'rating'])
    rec_df['movieId'] = rec_df['movieId'].astype(int)

    # Join with movie titles
    rec_df = rec_df.merge(movies, on='movieId', how='left')

    # Clip predicted ratings to valid range 0.5–5.0
    rec_df['rating'] = rec_df['rating'].clip(0.5, 5.0)

    # Separate movies with and without metadata
    rec_with_title    = rec_df.dropna(subset=['title'])
    rec_missing_title = rec_df[rec_df['title'].isna()]

    print(f"\nALS Top 10 for User {sample_user}")
    print("=" * 55)
    print(rec_with_title[['title', 'rating']]\
          .round(4).to_string(index=False))

    if len(rec_missing_title) > 0:
        print(f"\n⚠️  {len(rec_missing_title)} recommended "
              f"movies had no metadata")
        print(f"   movieIds: "
              f"{rec_missing_title['movieId'].tolist()}")
        print(f"   → ID alignment gap from Day 3 cleaning")
        print(f"   → CLIP embeddings (Day 12) will solve "
              f"this — visual retrieval needs no metadata")

    # Show user history for context
    print(f"\nUser {sample_user} rating history (top 5):")
    history = ratings[ratings['userId'] == sample_user]\
        .merge(movies, on='movieId', how='left')\
        .sort_values('rating', ascending=False)\
        .head(5)
    print(history[['title', 'rating']]\
          .to_string(index=False))

else:
    print(f"⚠️  No recommendations found for "
          f"User {sample_user}")




Generating ALS recommendations...


[Stage 193:===========================>                         (51 + 11) / 100]


ALS Top 10 for User 547
                                             title  rating
                                           The Kid     5.0
                               A Face in the Crowd     5.0
The Beatles: Eight Days a Week - The Touring Years     5.0
                                    Helter Skelter     5.0
                                   Willie and Phil     5.0
                                     The Staircase     5.0
                                    Mildred Pierce     5.0
                                         Peter Pan     5.0

⚠️  2 recommended movies had no metadata
   movieIds: [150856, 96075]
   → ID alignment gap from Day 3 cleaning
   → CLIP embeddings (Day 12) will solve this — visual retrieval needs no metadata

User 547 rating history (top 5):
                                             title  rating
The Beatles: Eight Days a Week - The Touring Years     5.0
                  The Treasure of the Sierra Madre     5.0
                                     

In [19]:
spark.stop()
print("\n✅ Spark session stopped")


✅ Spark session stopped


In [21]:
# Ranking Metrics For SVD

# Load CF results from Day 8 for comparison
# ── Metric functions defined directly here ────────────
# (no import needed — avoids module path issues)

def precision_at_k(recommended, relevant, k=10):
    top_k = recommended[:k]
    hits  = sum(1 for m in top_k if m in relevant)
    return hits / k if k > 0 else 0.0

def recall_at_k(recommended, relevant, k=10):
    if not relevant:
        return 0.0
    top_k = recommended[:k]
    hits  = sum(1 for m in top_k if m in relevant)
    return hits / len(relevant)

def ndcg_at_k(recommended, relevant, k=10):
    top_k = recommended[:k]
    dcg   = sum(
        1.0 / np.log2(i + 2)
        for i, m in enumerate(top_k)
        if m in relevant
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2)
               for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_surprise_ranking(
        model, ratings_df, movies_df,
        model_name: str,
        n_users: int = 200,
        k: int = 10) -> dict:
    """Evaluate ranking metrics for Surprise models"""

    precisions, recalls, ndcgs = [], [], []

    test_users = ratings_df['userId'].unique()
    eval_users = np.random.choice(
        test_users,
        size=min(n_users, len(test_users)),
        replace=False
    )

    all_movies = ratings_df['movieId'].unique()

    for user_id in eval_users:
        user_ratings = ratings_df[
            ratings_df['userId'] == user_id]

        if len(user_ratings) < 5:
            continue

        # Relevant = rated >= 4.0
        relevant = set(user_ratings[
            user_ratings['rating'] >= 4.0
        ]['movieId'].values)

        if not relevant:
            continue

        # Score all movies for this user
        scores = []
        for mid in all_movies:
            pred = model.predict(
                str(user_id), str(mid))
            scores.append((mid, pred.est))

        # Sort by predicted score descending
        scores.sort(key=lambda x: x[1],
                    reverse=True)
        rec_movies = [m for m, _ in scores[:k]]

        precisions.append(
            precision_at_k(rec_movies, relevant, k))
        recalls.append(
            recall_at_k(rec_movies, relevant, k))
        ndcgs.append(
            ndcg_at_k(rec_movies, relevant, k))

    return {
        "model":          model_name,
        f"precision@{k}": round(
            np.mean(precisions), 4),
        f"recall@{k}":    round(
            np.mean(recalls), 4),
        f"ndcg@{k}":      round(
            np.mean(ndcgs), 4),
    }


print("✅ Metric functions defined")
print("Computing ranking metrics for SVD...")
print("(This takes a few minutes — "
      "scoring all movies for 200 users)\n")

svd_ranking = evaluate_surprise_ranking(
    svd_best, ratings, movies, "SVD_tuned")

print(f"\nSVD Ranking Results:")
print(json.dumps(svd_ranking, indent=2))

✅ Metric functions defined
Computing ranking metrics for SVD...
(This takes a few minutes — scoring all movies for 200 users)


SVD Ranking Results:
{
  "model": "SVD_tuned",
  "precision@10": 0.0395,
  "recall@10": 0.0054,
  "ndcg@10": 0.0376
}
